# 50 — Blend geometry on the validated pool: extrinsic vs aligned vs frechet (bf16)

E2 (nb18) compared the three guide-averaging geometries and E2b (nb19)
mapped an aligned-vs-extrinsic crossover at rho ~ 0.33 — both on AWQ, the
legacy 15_e0 pairs, and metric v2 only. The quantization-impact report
showed the E2b low-rho aligned advantage was substantially an AWQ artifact
(bf16 gaps collapse to noise). This notebook re-asks the geometry question
under the current conventions: bf16, the 30 validated-pool pairs
(`43_pairs.json`), regular top-k CGD (K=4, steps=24, lambda=0), and all
three metric tracks — expression (v2), gauge-invariant angles (v3), and
perplexity.

Questions: (1) does ANY aligned-vs-extrinsic effect survive on clean
weights, in joint-z or in the angle track? (2) does frechet ~ extrinsic
(E2: 72% byte-identical texts) replicate on the validated pool?

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS, K, STEPS = 3, 4, 24
METHODS = ("extrinsic", "aligned", "frechet")
LAMBDAS = [0.0]
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"
SMOKE = False  # True -> 2 pairs x 2 prompts, artifacts get a _smoke suffix

PAIRS = json.loads(Path("resources/43_pairs.json").read_text())["pairs"]


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

if SMOKE:
    PAIRS, QUESTIONS = PAIRS[:2], QUESTIONS[:2]
SUFFIX = "_smoke" if SMOKE else ""
LOG_PATH = Path(f"resources/50_blend_geometry_bf16{SUFFIX}.jsonl")
LOG_PATH.unlink(missing_ok=True)
SUMMARY_PATH = Path(f"resources/50_blend_geometry_summary{SUFFIX}.csv")
GAP_PLOT_PATH = Path(f"resources/50_blend_geometry_gap_vs_rho{SUFFIX}.png")
BAR_PLOT_PATH = Path(f"resources/50_blend_geometry_summary{SUFFIX}.png")
print(
    f"{len(PAIRS)} pairs x {len(METHODS)} methods x {len(LAMBDAS)} lambda "
    f"x {len(QUESTIONS)} prompts = "
    f"{len(PAIRS) * len(METHODS) * len(LAMBDAS) * len(QUESTIONS)} records"
)

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    device_map="auto", id=MODEL_ID, torch_dtype=torch.bfloat16
)
prompts = [chat(u, a) for u, a in QUESTIONS]
harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
# force the one-time cuDF WordNet parquet load (~300 MB on-GPU) now,
# while memory is free - inside the generation loop it can OOM
_ = harness.member_lemmas(PAIRS[0]["a"])
baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)
print(f"{len(baseline_texts)} baselines")

Pre-flight: the intrinsic means (aligned, frechet) require equal effective
ranks across the pair — `Concept.average` raises otherwise — and
`frechet_mean` can in principle fail to converge for very distant frames.
All guides are built up front so any failure is a loud `EXCLUDED` line
here, never a mid-run crash; an excluded pair drops from all three methods,
keeping the comparison paired. Expected on this pool: 30/30 ready (all
pool concepts have effective rank 3 at l11/t3).

In [ ]:
pairs = []
for p in PAIRS:
    concept_a = fur.get_concept_cached(p["a"], MIN_LEMMAS, MAX_TOKENS)
    concept_b = fur.get_concept_cached(p["b"], MIN_LEMMAS, MAX_TOKENS)
    try:
        guides = {
            method: Concept.average([concept_a, concept_b], method=method)
            for method in METHODS
        }
    except (ValueError, RuntimeError) as error:
        print(f"EXCLUDED {p['a']} / {p['b']}: {error}")
        continue
    pairs.append(
        {
            "a": p["a"],
            "b": p["b"],
            "rho_pool": p["rho"],
            "rho": float(concept_a.rho(concept_b)),
            "concept_a": concept_a,
            "concept_b": concept_b,
            "guides": guides,
        }
    )
print(f"{len(pairs)}/{len(PAIRS)} pairs ready")

In [ ]:
for pair in pairs:
    name_a, name_b = pair["a"], pair["b"]
    concepts = {name_a: pair["concept_a"], name_b: pair["concept_b"]}
    for method in METHODS:
        for lam in LAMBDAS:
            torch.cuda.empty_cache()
            # single blended guide through the unified loop; the canonical
            # single-guide wrapper has no fluency dial (nb44 precedent)
            texts, probe = fur.generate_with_topk_multi_guide(
                prompts,
                guides=[pair["guides"][method]],
                k=K,
                steps=STEPS,
                fluency_weight=lam,
            )
            harness.evaluate(
                prompts,
                texts,
                [name_a, name_b],
                config={
                    "model": MODEL_KEY,
                    "method": method,
                    "lam": lam,
                    "a": name_a,
                    "b": name_b,
                    "pair": f"{name_a} / {name_b}",
                    "rho": pair["rho"],
                    "rho_pool": pair["rho_pool"],
                    "k": K,
                    "steps": STEPS,
                },
                probe=probe,
                baseline_texts=baseline_texts,
                concepts=concepts,
                angles=True,
            )
    print(f"rho={pair['rho']:.3f} {name_a} / {name_b}: done")

In [ ]:
parsed = [json.loads(line) for line in LOG_PATH.read_text().strip().split("\n")]
assert len(parsed) == len(pairs) * len(METHODS) * len(LAMBDAS) * len(prompts)
assert all(
    r["expression"] is not None and r["angles"] is not None for r in parsed
), "metric v2 + v3 present everywhere"
assert not any(
    np.isnan(v)
    for r in parsed
    for s in (r["config"]["a"], r["config"]["b"])
    for v in r["angles"][s]["delta"].values()
), "NaN in angle deltas"
print(f"gate: {len(parsed)} records parse, expression + angles logged")

Analysis conventions. `joint_z`: per-concept expression deltas are z-scored
over ALL of that concept's records in this log (pooled across method and
pair side — the E2b convention), then min over the pair per record.
`joint_cos`: min of the two raw `token_cos` deltas — bounded and
scale-comparable, so no z-scoring (the nb38/43 convention). The geodesic
and mcc tracks are logged per record but not aggregated here.

In [ ]:
rows = []
for record in parsed:
    config = record["config"]
    expr, ang = record["expression"], record["angles"]
    rows.append(
        {
            "pair": config["pair"],
            "rho": config["rho"],
            "method": config["method"],
            "lam": config["lam"],
            "prompt": record["prompt"],
            "concept_a": config["a"],
            "concept_b": config["b"],
            "delta_a": expr[config["a"]]["delta"],
            "delta_b": expr[config["b"]]["delta"],
            "d_cos_a": ang[config["a"]]["delta"]["token_cos"],
            "d_cos_b": ang[config["b"]]["delta"]["token_cos"],
            "ppl_ratio": record["ppl_ratio"],
            "fluency_flag": record["fluency_flag"],
        }
    )
results = pd.DataFrame(rows)

# per-concept z across ALL records of that concept, regardless of pair or side
long = pd.concat(
    [
        results[["concept_a", "delta_a"]].rename(
            columns={"concept_a": "concept", "delta_a": "delta"}
        ),
        results[["concept_b", "delta_b"]].rename(
            columns={"concept_b": "concept", "delta_b": "delta"}
        ),
    ],
    keys=["a", "b"],
)
stats = long.groupby("concept")["delta"].agg(["mean", "std"])
for side in ("a", "b"):
    mu = results[f"concept_{side}"].map(stats["mean"])
    sd = results[f"concept_{side}"].map(stats["std"])
    results[f"z_{side}"] = (results[f"delta_{side}"] - mu) / (sd + 1e-8)
results["joint_z"] = results[["z_a", "z_b"]].min(axis=1)
results["joint_cos"] = results[["d_cos_a", "d_cos_b"]].min(axis=1)
results["both_improved"] = results[["delta_a", "delta_b"]].min(axis=1) > 0
results["both_improved_cos"] = results["joint_cos"] > 0

summary = (
    results.groupby(["method", "lam"])
    .agg(
        joint_z=("joint_z", "mean"),
        joint_cos=("joint_cos", "mean"),
        both_improved=("both_improved", "mean"),
        both_improved_cos=("both_improved_cos", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        flag_share=("fluency_flag", "mean"),
        n=("joint_z", "size"),
    )
    .reset_index()
)
summary.to_csv(SUMMARY_PATH, index=False)
summary.round(4)

In [ ]:
def crossover_fit(sub, track):
    """Per-pair paired aligned-extrinsic gaps and a weighted linear fit vs rho."""
    wide = sub.pivot_table(
        index=["pair", "rho", "prompt"], columns="method", values=track
    ).reset_index()
    wide["gap"] = wide["aligned"] - wide["extrinsic"]
    gaps = (
        wide.groupby(["pair", "rho"])["gap"]
        .agg(["mean", "sem", "count"])
        .reset_index()
        .sort_values("rho")
    )
    if len(gaps) < 4:
        return gaps, None, None, "too few pairs for a fit"
    # byte-identical texts give sem = 0 -> infinite weight; clip to smallest nonzero
    positive = gaps["sem"][gaps["sem"] > 0]
    floor = float(positive.min()) if len(positive) else 1e-9
    coeffs, cov = np.polyfit(
        gaps["rho"],
        gaps["mean"],
        deg=1,
        w=1 / gaps["sem"].clip(lower=floor),
        cov=True,
    )
    slope_se = float(np.sqrt(cov[0, 0]))
    crossover = -coeffs[1] / coeffs[0]
    in_range = gaps["rho"].min() <= crossover <= gaps["rho"].max()
    if abs(coeffs[0]) > slope_se and in_range:
        verdict = f"crossover at rho = {crossover:.3f}"
    else:
        verdict = (
            f"no crossover within rho in [{gaps['rho'].min():.2f}, "
            f"{gaps['rho'].max():.2f}] (slope consistent with 0)"
        )
    return gaps, coeffs, slope_se, verdict


fits = {}
for lam in LAMBDAS:
    for track in ("joint_z", "joint_cos"):
        gaps, coeffs, slope_se, verdict = crossover_fit(
            results[results["lam"] == lam], track
        )
        fits[(lam, track)] = (gaps, coeffs, verdict)
        if coeffs is None:
            print(f"[lam={lam} {track}] {verdict}")
        else:
            print(
                f"[lam={lam} {track}] gap = {coeffs[0]:+.3f} * rho {coeffs[1]:+.3f} "
                f"(slope SE {slope_se:.3f}); {verdict}"
            )

In [ ]:
for lam in LAMBDAS:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    labels = {
        "joint_z": "joint-z gap (aligned - extrinsic)",
        "joint_cos": "joint token-cos gap (aligned - extrinsic)",
    }
    for ax, track in zip(axes[:2], ("joint_z", "joint_cos")):
        gaps, coeffs, verdict = fits[(lam, track)]
        ax.errorbar(gaps["rho"], gaps["mean"], yerr=gaps["sem"], fmt="o", capsize=3)
        if coeffs is not None:
            fit_x = np.linspace(gaps["rho"].min(), gaps["rho"].max(), 50)
            ax.plot(fit_x, np.polyval(coeffs, fit_x), "k--", lw=1)
        ax.axhline(0, color="gray", lw=0.5)
        ax.set_xlabel("rho")
        ax.set_ylabel(labels[track])
        ax.set_title(f"lam={lam}: {verdict}", fontsize=9)

    ppl = (
        results[results["lam"] == lam]
        .groupby(["rho", "method"])["ppl_ratio"]
        .mean()
        .unstack()
    )
    markers = {"extrinsic": "o-", "aligned": "^-", "frechet": "s-"}
    for method in METHODS:
        axes[2].plot(ppl.index, ppl[method], markers[method], label=method, ms=4)
    axes[2].axhline(2.5, color="gray", ls="--", lw=0.8)
    axes[2].set_xlabel("rho")
    axes[2].set_ylabel("mean PPL ratio")
    axes[2].set_title("fluency cost")
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    path = (
        GAP_PLOT_PATH
        if len(LAMBDAS) == 1
        else GAP_PLOT_PATH.with_stem(f"{GAP_PLOT_PATH.stem}_lam{lam}")
    )
    plt.savefig(path, dpi=150)
    plt.show()

In [ ]:
texts_wide = (
    pd.DataFrame(
        {
            "pair": [r["config"]["pair"] for r in parsed],
            "lam": [r["config"]["lam"] for r in parsed],
            "prompt": [r["prompt"] for r in parsed],
            "method": [r["config"]["method"] for r in parsed],
            "text": [r["text"] for r in parsed],
        }
    )
    .pivot_table(
        index=["pair", "lam", "prompt"],
        columns="method",
        values="text",
        aggfunc="first",
    )
    .reset_index()
)
for lam in LAMBDAS:
    sub = texts_wide[texts_wide["lam"] == lam]
    print(
        f"[lam={lam}] identical texts: "
        f"frechet==extrinsic {(sub['frechet'] == sub['extrinsic']).mean():.1%}, "
        f"aligned==extrinsic {(sub['aligned'] == sub['extrinsic']).mean():.1%} "
        f"(E2 AWQ anchor: 72% frechet==extrinsic)"
    )
for track in ("joint_z", "joint_cos"):
    wide = results.pivot_table(
        index=["pair", "lam", "prompt"], columns="method", values=track
    ).reset_index()
    for other in ("frechet", "aligned"):
        gap = wide[other] - wide["extrinsic"]
        print(
            f"{track} {other}-extrinsic: mean {gap.mean():+.4f}, "
            f"mean |gap| {gap.abs().mean():.4f}"
        )

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
titles = {
    "joint_cos": "mean joint token-cos delta",
    "both_improved_cos": "both-improved rate (cos)",
    "ppl_ratio": "mean PPL ratio",
}
for ax, col in zip(axes, titles):
    piv = summary.pivot_table(index="method", columns="lam", values=col)
    piv.loc[list(METHODS)].plot.bar(ax=ax, legend=(ax is axes[0]))
    ax.set_title(titles[col])
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(BAR_PLOT_PATH, dpi=150)
plt.show()

In [ ]:
def answer(text: str) -> str:
    return text.split("assistant<|end_header_id|>")[-1]


target = next((q for q, _ in QUESTIONS if "short story" in q), QUESTIONS[-1][0])
ordered = sorted(pairs, key=lambda p: p["rho"])
for pair in (ordered[0], ordered[len(ordered) // 2], ordered[-1]):
    key = f"{pair['a']} / {pair['b']}"
    for method in METHODS:
        example = next(
            r
            for r in parsed
            if r["config"]["pair"] == key
            and r["config"]["method"] == method
            and r["config"]["lam"] == LAMBDAS[0]
            and target in r["prompt"]
        )
        deltas = {
            name.split(".")[0]: round(v["delta"], 4)
            for name, v in example["expression"].items()
        }
        print(f"[rho={pair['rho']:.3f} {key} / {method}] deltas={deltas}")
        print("   ", answer(example["text"])[:130])
    print()

In [ ]:
if SMOKE:
    print("SMOKE RUN — _smoke artifacts only; set SMOKE = False and rerun")